# ESPHome microWakeWord Training

This notebook generates an ESPHome-compatible `micro_wake_word` model and manifest.
It replaces the `openwakeword` + ONNX + `onnx2tf` flow with the official `micro-wake-word` training/export pipeline.

In [ ]:
# 1. Test one generated pronunciation

import os
import sys
import importlib.util
from functools import wraps
from IPython.display import Audio

if not os.path.exists("./piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator
    !cd piper-sample-generator && git checkout 213d4d5

model_path = "piper-sample-generator/models/en_US-libritts_r-medium.pt"
if not os.path.exists(model_path):
    !wget -O {model_path} "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"

!pip install -q piper-tts piper-phonemize-cross webrtcvad

if importlib.util.find_spec("torch") is None:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

import torch

# Piper expects the pre-2.6 torch.load default for this trusted checkpoint.
_original_torch_load = torch.load

@wraps(_original_torch_load)
def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)

torch.load = _torch_load_compat

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")

from generate_samples import generate_samples

target_word = "hey___ee__gore"  # @param {type:"string"}

def text_to_speech(text):
    generate_samples(
        text=text,
        max_samples=1,
        length_scales=[1.1],
        noise_scales=[0.7],
        noise_scale_ws=[0.7],
        output_dir="./",
        batch_size=1,
        auto_reduce_batch_size=True,
        file_names=["test_generation.wav"],
    )

text_to_speech(target_word)
Audio("test_generation.wav", autoplay=True)


In [3]:
# @title
# 2. Install microWakeWord and dependencies

import locale
import os
import importlib.util

def getpreferredencoding(do_setlocale=True):
    return "UTF-8"

locale.getpreferredencoding = getpreferredencoding

!apt-get -qq update
!apt-get -qq install -y ffmpeg git-lfs

!pip install -q "git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f"
!pip install -q datasets scipy tqdm pyyaml

if importlib.util.find_spec("torch") is None:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

if not os.path.exists("./micro-wake-word"):
    !git clone https://github.com/OHF-Voice/micro-wake-word

!pip install -q -e ./micro-wake-word


  Cloning https://github.com/whatsnowplaying/audio-metadata (to revision d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f) to /tmp/pip-req-build-9qzqs1j7
  Running command git clone --filter=blob:none --quiet https://github.com/whatsnowplaying/audio-metadata /tmp/pip-req-build-9qzqs1j7
  Running command git rev-parse -q --verify 'sha^d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
  Running command git fetch -q https://github.com/whatsnowplaying/audio-metadata d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Running command git checkout -q d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Resolved https://github.com/whatsnowplaying/audio-metadata to commit d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
fatal: destination path 'micro-wake-word' already exists and is not an empty directory.
Obtaining file:///content/micro-wake-word
  Installing build dependencies ... done
  Checking if b

In [4]:
# Imports

import json
import sys
import shutil
import datasets
import numpy as np
import scipy
import yaml
from pathlib import Path
from tqdm import tqdm

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")
from generate_samples import generate_samples


In [6]:
# Download RIR dataset

import os
import shutil
from pathlib import Path
from tqdm import tqdm

output_dir = Path("./mit_rirs")
source_dir = Path("./MIT_environmental_impulse_responses/16khz")

output_dir.mkdir(exist_ok=True)

if not source_dir.exists():
    !git lfs install
    !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses

rir_files = list(source_dir.glob("*.wav"))
if not rir_files:
    raise RuntimeError(f"No RIR wav files found in {source_dir}")

existing_rirs = len(list(output_dir.glob("*.wav")))
if existing_rirs != len(rir_files):
    for src in tqdm(rir_files):
        shutil.copy2(src, output_dir / src.name)

print(f"mit_rirs: {len(list(output_dir.glob('*.wav')))} wav files")


In [20]:
import subprocess
from pathlib import Path
from tqdm import tqdm

def convert_audio_tree(src_root, pattern, out_dir):
    src_root = Path(src_root)
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)

    files = sorted(src_root.glob(pattern))
    if not files:
        raise RuntimeError(f"No source files found in {src_root} matching {pattern}")

    for src in tqdm(files):
        rel_stem = src.relative_to(src_root).with_suffix("")
        dst_name = "__".join(rel_stem.parts) + ".wav"
        dst = out_dir / dst_name
        if dst.exists():
            continue
        subprocess.run(
            ["ffmpeg", "-y", "-i", str(src), "-ac", "1", "-ar", "16000", str(dst)],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )

    count = len(list(out_dir.glob("*.wav")))
    print(f"{out_dir}: {count} wav files")


In [ ]:
# Download AudioSet sample and convert to 16 kHz wav

import os
import subprocess

import datasets
from tqdm import tqdm

output_dir = Path("./audioset_16k")
output_dir.mkdir(exist_ok=True)
tmp_dir = Path("./audioset_tmp")
tmp_dir.mkdir(exist_ok=True)

max_examples = 2000  # @param {type:"slider", min:500, max:10000, step:500}

audioset = datasets.load_dataset(
    "agkphysics/AudioSet",
    "balanced",
    split=f"train[:{max_examples}]",
)

# Avoid torchcodec; work with raw paths/bytes from parquet instead.
audioset = audioset.cast_column("audio", datasets.Audio(decode=False))

for row in tqdm(audioset):
    audio = row["audio"]
    dst = output_dir / f"{row['video_id']}.wav"
    if dst.exists():
        continue

    src_path = None
    if audio.get("path") and os.path.exists(audio["path"]):
        src_path = audio["path"]
    elif audio.get("bytes") is not None:
        tmp_src = tmp_dir / f"{row['video_id']}.flac"
        tmp_src.write_bytes(audio["bytes"])
        src_path = str(tmp_src)

    if src_path is None:
        continue

    subprocess.run(
        ["ffmpeg", "-y", "-i", src_path, "-ac", "1", "-ar", "16000", str(dst)],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

print({"audioset_16k": len(list(output_dir.glob("*.wav")))})


In [ ]:
# Download FMA xsmall sample and convert to 16 kHz wav

import os
download_dir = "./fma"
if not os.path.exists(download_dir):
    os.mkdir(download_dir)
    fname = "fma_xs.zip"
    link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
    out_path = f"{download_dir}/{fname}"
    !wget -O {out_path} {link}
    !cd {download_dir} && unzip -q {fname}

print(f"FMA source files: {len(list(Path('fma/fma_small').glob('**/*.mp3')))}")
convert_audio_tree("fma/fma_small", "**/*.mp3", "fma_16k")
print({"fma_16k": len(list(Path("fma_16k").glob("*.wav")))})


In [14]:
# Generate positive training samples with Piper

os.makedirs("generated_samples", exist_ok=True)

number_of_examples = 3000  # @param {type:"slider", min:500, max:20000, step:500}

generate_samples(
    text=target_word,
    max_samples=number_of_examples,
    output_dir="generated_samples",
    batch_size=32,
    auto_reduce_batch_size=True,
)


/content/piper-sample-generator/generate_samples.py:76: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch_model = torch.load(model_path)


In [16]:
import os
import site
import importlib.util

if importlib.util.find_spec("microwakeword") is None:
    if not os.path.exists("./micro-wake-word"):
        !git clone https://github.com/OHF-Voice/micro-wake-word
    !pip install -q -e ./micro-wake-word

# Make sure the current kernel sees the editable install
site.main()

import microwakeword
print("microwakeword import OK:", microwakeword.__file__)


Obtaining file:///content/micro-wake-word
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for microwakeword (pyproject.toml) ... done
  Created wheel for microwakeword: filename=microwakeword-0.1.0-0.editable-py3-none-any.whl size=9935 sha256=120e76010d2a092cd71b2ab3d4c6d094873a8c76e59cebb0b259b82c6f642c21
  Stored in directory: /tmp/pip-ephem-wheel-cache-eob266oi/wheels/04/95/30/9f826a946450c24b8307db2173af7f46d58ca991e15aa5d776
Successfully built microwakeword
  Attempting uninstall: microwakeword
    Found existing installation: microwakeword 0.1.0
    Uninstalling microwakeword-0.1.0:
      Successfully uninstalled microwakeword-0.1.0
microwakeword import OK: /content/micro-wake-word/microwakeword/__init__.py


In [18]:
from pathlib import Path

print("fma_16k exists:", Path("fma_16k").exists())
print("audioset_16k exists:", Path("audioset_16k").exists())

print("fma_16k wavs:", len(list(Path("fma_16k").glob("*.wav"))))
print("audioset_16k wavs:", len(list(Path("audioset_16k").glob("*.wav"))))

print("sample fma:", list(Path("fma_16k").glob("*.wav"))[:5])
print("sample audioset:", list(Path("audioset_16k").glob("*.wav"))[:5])


fma_16k exists: True
audioset_16k exists: True
fma_16k wavs: 0
audioset_16k wavs: 0
sample fma: []
sample audioset: []


In [ ]:
# Build ESPHome-compatible features

from pathlib import Path
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips_settings = {
    "input_directory": "generated_samples",
    "file_pattern": "*.wav",
    "max_clip_duration_s": None,
    "remove_silence": False,
    "random_split_seed": 10,
    "split_count": 0.1,
}

background_counts = {
    "fma_16k": len(list(Path("fma_16k").rglob("*.wav"))),
    "audioset_16k": len(list(Path("audioset_16k").rglob("*.wav"))),
}
if not all(background_counts.values()):
    raise RuntimeError(f"Background audio is missing. Counts: {background_counts}")

augmentation_settings = {
    "augmentation_duration_s": 3.2,
    "augmentation_probabilities": {
        "SevenBandParametricEQ": 0.1,
        "TanhDistortion": 0.1,
        "PitchShift": 0.1,
        "BandStopFilter": 0.1,
        "AddColorNoise": 0.1,
        "AddBackgroundNoise": 0.75,
        "Gain": 1.0,
        "RIR": 0.5,
    },
    "impulse_paths": ["mit_rirs"],
    "background_paths": ["fma_16k", "audioset_16k"],
    "background_min_snr_db": -5,
    "background_max_snr_db": 10,
    "min_jitter_s": 0.195,
    "max_jitter_s": 0.205,
}

spectrogram_generation_settings = {
    "slide_frames": 10,
    "step_ms": 10,
}

# Smoke-test the current API and record the settings for training_config below.
clips = Clips(**clips_settings)
augmenter = Augmentation(**augmentation_settings)
spectrograms = SpectrogramGeneration(
    clips=clips,
    augmenter=augmenter,
    **spectrogram_generation_settings,
)

sample_spectrogram = next(spectrograms.spectrogram_generator(random=True, max_clips=1))
print({
    "background_counts": background_counts,
    "sample_spectrogram_shape": sample_spectrogram.shape,
})


In [ ]:
# Write training config

config = {}
config["window_step_ms"] = 10
config["train_dir"] = "trained_models/hey_igor"
config["features"] = [
    {
        "type": "clips",
        "truth": True,
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truncation_strategy": "truncate_start",
        "clips_settings": clips_settings,
        "augmentation_settings": augmentation_settings,
        "spectrogram_generation_settings": spectrogram_generation_settings,
    },
    {
        "features_dir": "fma_16k",
        "sampling_weight": 8.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "audio",
    },
    {
        "features_dir": "audioset_16k",
        "sampling_weight": 8.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "audio",
    },
]
config["training_steps"] = [10000]
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]
config["learning_rates"] = [0.001]
config["batch_size"] = 128
config["time_mask_max_size"] = [0]
config["time_mask_count"] = [0]
config["freq_mask_max_size"] = [0]
config["freq_mask_count"] = [0]
config["eval_step_interval"] = 500
config["clip_duration_ms"] = 1500
config["target_minimization"] = 0.9
config["minimization_metric"] = None
config["maximization_metric"] = "average_viable_recall"

with open("training_parameters.yaml", "w") as f:
    yaml.dump(config, f)


In [ ]:
# Train and export streaming quantized TFLite model

!python -m microwakeword.model_train_eval \
  --training_config=training_parameters.yaml \
  --train 1 \
  --restore_checkpoint 1 \
  --test_tf_nonstreaming 0 \
  --test_tflite_nonstreaming 0 \
  --test_tflite_nonstreaming_quantized 0 \
  --test_tflite_streaming 0 \
  --test_tflite_streaming_quantized 1 \
  --use_weights best_weights \
  mixednet \
  --pointwise_filters "64,64,64,64" \
  --repeat_in_block "1,1,1,1" \
  --mixconv_kernel_sizes "[5],[7,11],[9,15],[23]" \
  --residual_connection "0,0,0,0" \
  --first_conv_filters 32 \
  --first_conv_kernel_size 5 \
  --stride 3


In [ ]:
# Package ESPHome model files

from google.colab import files

shutil.copy(
    "trained_models/hey_igor/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite",
    "hey_igor.tflite",
)

manifest = {
    "type": "micro",
    "wake_word": "Hey Igor",
    "author": "Alex",
    "website": "https://github.com/cajo-dk/microwakeword",
    "model": "hey_igor.tflite",
    "trained_languages": ["en"],
    "version": 2,
    "micro": {
        "probability_cutoff": 0.97,
        "sliding_window_size": 5,
        "feature_step_size": 10,
        "tensor_arena_size": 262144,
        "minimum_esphome_version": "2024.7"
    }
}

with open("hey_igor.json", "w") as f:
    json.dump(manifest, f, indent=2)

files.download("hey_igor.tflite")
files.download("hey_igor.json")
